# Sequential models: results and sensitivity (Role C: Arshad)
Goal: compare LSTM / BiLSTM / GRU with the TF-IDF baseline (Phase 1), then show how much each
BiLSTM setting matters (Phase 2). Reads the run files in `results/runs/`; no training here.
Figures are saved to `results/figures/seq_*.png` for the slides.

## Setup

In [ ]:
#Setup
import sys, pathlib, json
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # repo root, so `src` imports work

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from src.config import FIGURES_DIR, RUNS_DIR

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
AUTHOR = "Arshad"

# one colour per model, the same in every chart; the TF-IDF baseline is grey
COLORS = {"bilstm": "#2a78d6", "lstm": "#eb6834", "gru": "#1baf7a", "tfidf": "#898781"}
INK, MUTED = "#0b0b0b", "#52514e"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 10,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlecolor": INK,
    "lines.linewidth": 2, "lines.markersize": 8,
})
BASELINE_HANDLE = Line2D([], [], color=COLORS["tfidf"], ls="--", lw=1.5, label="TF-IDF + LR baseline")

In [ ]:
#Load every val run into one table
runs = pd.json_normalize([json.loads(p.read_text(encoding="utf-8")) for p in RUNS_DIR.glob("*.json")])
runs = runs[runs["split"] == "val"]
tfidf = runs[runs["model"] == "tfidf-lr-sanity"].set_index("dataset")

seq = runs[(runs["author"] == AUTHOR) & runs["model"].isin(["lstm", "bilstm", "gru"])].copy()
seq["params.truncation"] = seq["params.truncation"].fillna("head")   # Phase 1 runs predate the option

DEFAULT_LEN = {"isot": 512, "liar": 64, "liar6": 64, "covid": 128}
METRIC = {"isot": "f1_macro", "liar": "f1_macro", "liar6": "accuracy", "covid": "f1_macro"}

def is_default(r):
    return bool(r["params.hidden_size"] == 128 and r["params.dropout"] == 0.3
                and r["params.freeze_embeddings"] and r["params.truncation"] == "head"
                and r["params.max_len"] == DEFAULT_LEN.get(r["dataset"]))

seq["default"] = seq.apply(is_default, axis=1)
# Phase 1 = default settings on the 4 main tasks (earliest run, in case of repeats)
phase1 = (seq[seq["default"]].sort_values("timestamp").groupby(["model", "dataset"]).head(1))
print(f"{len(seq)} sequential val runs, {len(phase1)} of them Phase 1")

## 1. Phase 1: every model vs the TF-IDF baseline

In [ ]:
#Plot 1, Phase 1 results (val)
TASKS = [("isot", "ISOT (articles)"), ("liar", "LIAR (claims)"),
         ("liar6", "LIAR-6 (6 levels)"), ("covid", "COVID (posts)")]
MODELS = [("lstm", "LSTM"), ("bilstm", "BiLSTM"), ("gru", "GRU")]

fig, axes = plt.subplots(1, 4, figsize=(13, 2.9))
for ax, (task, title) in zip(axes, TASKS):
    metric = METRIC[task]
    base = tfidf.loc[task, metric]
    vals = [phase1.query("model == @m and dataset == @task")[metric].iloc[0] for m, _ in MODELS]
    ax.axvline(base, color=COLORS["tfidf"], ls="--", lw=1.5)
    for y, ((m, _), v) in enumerate(zip(MODELS, vals)):
        ax.plot(v, y, "o", color=COLORS[m], ms=9)
        ax.annotate(f"{v:.3f}", (v, y), xytext=(0, 8), textcoords="offset points",
                    ha="center", fontsize=8, color=INK)
    lo, hi = min(vals + [base]), max(vals + [base])
    span = max(hi - lo, 0.01)
    ax.set_xlim(lo - 0.25 * span, hi + 0.25 * span)
    ax.set_yticks(range(len(MODELS)), [name for _, name in MODELS])
    ax.set_ylim(len(MODELS) - 0.4, -0.7)
    ax.grid(axis="y", visible=False)
    ax.set_title(title, loc="left")
    ax.set_xlabel("val accuracy" if metric == "accuracy" else "val macro-F1")
fig.legend(handles=[BASELINE_HANDLE], loc="upper right", frameon=False, fontsize=9)
fig.suptitle("Phase 1: sequential models vs TF-IDF (default settings)", x=0.01, ha="left",
             fontweight="bold", color=INK)
fig.tight_layout(rect=(0, 0, 1, 0.93))
fig.savefig(FIGURES_DIR / "seq_phase1_models.png", dpi=150)

**Takeaway:** BiLSTM is the best sequential model (best on ISOT, LIAR-6 and COVID). It clearly beats
TF-IDF on COVID (0.955 vs 0.924); on LIAR all three models are about level with TF-IDF.
**In plain terms:** reading words in order helps on social-media posts, but not on short political claims.

## 2. Phase 2: how many words to read (max length)
BiLSTM, one setting changed at a time. The circled point is the Phase 1 default.

In [ ]:
#Helpers for the sensitivity plots
YLIM = {"liar": (0.595, 0.640), "isot": (0.978, 1.0015)}

def sweep(dataset, group):
    """BiLSTM runs for one setting on one dataset, plus the Phase 1 default point."""
    b = seq[(seq["model"] == "bilstm") & (seq["dataset"] == dataset)]
    return pd.concat([b[b["notes"] == f"sens: {group}"], b[b["default"]].sort_values("timestamp").head(1)])

def num(v):
    return f"{v:g}"   # 128.0 -> "128", 0.1 -> "0.1"

def sens_panel(ax, d, col, fmt=num, title="", y="f1_macro", ylabel="val macro-F1",
               base=None, ascending=True, value_fmt="{:.3f}"):
    d = d.sort_values(col, ascending=ascending)
    xs = list(range(len(d)))
    ax.plot(xs, d[y], "-o", color=COLORS["bilstm"])
    for x, (_, r) in zip(xs, d.iterrows()):
        if r["default"]:
            ax.plot(x, r[y], "o", ms=15, mfc="none", mec=INK, mew=1.2)   # ring = Phase 1 default
        ax.annotate(value_fmt.format(r[y]), (x, r[y]), xytext=(0, 10), textcoords="offset points",
                    ha="center", fontsize=8, color=INK)
    if base is not None:
        ax.axhline(base, color=COLORS["tfidf"], ls="--", lw=1.5)
    ax.set_xticks(xs, [fmt(v) for v in d[col]])
    ax.set_xlim(-0.4, len(d) - 0.6)
    ax.grid(axis="x", visible=False)
    ax.set_title(title, loc="left")
    ax.set_ylabel(ylabel)

In [ ]:
#Plot 2, max length
isot_len, liar_len = sweep("isot", "max_len"), sweep("liar", "max_len")
isot_len = isot_len.assign(minutes=isot_len["runtime_s"] / 60)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
sens_panel(axes[0], isot_len, "params.max_len", title="ISOT: score", base=tfidf.loc["isot", "f1_macro"])
axes[0].set_ylim(*YLIM["isot"])
sens_panel(axes[1], isot_len, "params.max_len", title="ISOT: training time", y="minutes",
           ylabel="minutes per run", value_fmt="{:.1f}")
axes[1].set_ylim(0, isot_len["minutes"].max() * 1.2)
sens_panel(axes[2], liar_len, "params.max_len", title="LIAR: score", base=tfidf.loc["liar", "f1_macro"])
axes[2].set_ylim(*YLIM["liar"])
for ax in axes:
    ax.set_xlabel("max length (words read)")
fig.legend(handles=[BASELINE_HANDLE], loc="upper right", frameon=False, fontsize=9)
fig.suptitle("Max length: ISOT keeps improving, LIAR doesn't care (BiLSTM, val)", x=0.01, ha="left",
             fontweight="bold", color=INK)
fig.tight_layout(rect=(0, 0, 1, 0.93))
fig.savefig(FIGURES_DIR / "seq_sens_max_len.png", dpi=150)

**Takeaway:** on ISOT, reading more words helps: 0.987 at 128 → 0.997 at 512. The gain flattens after
256, and 1024 words takes ~3× longer for +0.001, so 512 is the sweet spot. LIAR claims are ~22 words,
so a longer limit only adds padding and the score stays flat.
**In plain terms:** long articles need the model to read further; short claims are already read in full.

## 3. Phase 2: memory size, dropout and embeddings

In [ ]:
#Plot 3, hidden size / dropout / embeddings
SETTINGS = [("hidden", "params.hidden_size", num, "Hidden size", True),
            ("dropout", "params.dropout", num, "Dropout", True),
            ("embeddings", "params.freeze_embeddings", lambda v: "frozen" if v else "trainable",
             "GloVe embeddings", False)]

fig, axes = plt.subplots(2, 3, figsize=(13, 6.2))
for row, (dataset, name) in enumerate([("liar", "LIAR"), ("isot", "ISOT")]):
    for col, (group, param, fmt, title, ascending) in enumerate(SETTINGS):
        ax = axes[row, col]
        sens_panel(ax, sweep(dataset, group), param, fmt=fmt, title=f"{name}: {title}",
                   base=tfidf.loc[dataset, "f1_macro"], ascending=ascending)
        ax.set_ylim(*YLIM[dataset])
fig.legend(handles=[BASELINE_HANDLE], loc="upper right", frameon=False, fontsize=9)
fig.suptitle("Other settings barely matter (BiLSTM, val)", x=0.01, ha="left", fontweight="bold", color=INK)
fig.tight_layout(rect=(0, 0, 1, 0.95))
fig.savefig(FIGURES_DIR / "seq_sens_settings.png", dpi=150)

**Takeaway:** LIAR stays at 0.611–0.619 for every setting except dropout 0.5 (0.628). LIAR's val set is
only 1,284 claims, so ~0.01 differences are within run-to-run noise. ISOT is at the ceiling
(0.996–0.998) for every setting; 0.001 ≈ 4 of 3,823 articles.
**In plain terms:** tuning doesn't fix LIAR. A short claim doesn't contain the evidence of whether
it's true, so the limit is the information, not the model.

## 4. ISOT checks: where does the near-perfect score come from?
Same articles and splits each time; only the text changes (variants from Role A's PR #2).

In [ ]:
#Plot 4, ISOT checks, BiLSTM vs TF-IDF
def bilstm_val(dataset, notes):
    d = seq[(seq["model"] == "bilstm") & (seq["dataset"] == dataset)]
    d = d[d["default"]] if notes is None else d[d["notes"] == notes]
    return d.sort_values("timestamp")["f1_macro"].iloc[0]

CHECKS = [  # label, BiLSTM score, TF-IDF score (None = not run)
    ("Default\n(start of article)", bilstm_val("isot", None), tfidf.loc["isot", "f1_macro"]),
    ("Web clutter removed\n(isot_nostyle)", bilstm_val("isot_nostyle", "check: nostyle"),
     tfidf.loc["isot_nostyle", "f1_macro"]),
    ("Start + end\nof article", bilstm_val("isot", "check: truncate"), None),
    ("Headline added\n(isot_title)", bilstm_val("isot_title", "check: title"),
     tfidf.loc["isot_title", "f1_macro"]),
]

fig, ax = plt.subplots(figsize=(8, 3.6))
for y, (label, b, t) in enumerate(CHECKS):
    if t is not None:
        ax.plot([t, b], [y, y], color="#c3c2b7", lw=1.5, zorder=1)
        ax.plot(t, y, "o", color=COLORS["tfidf"], ms=9, zorder=2)
        ax.annotate(f"{t:.3f}", (t, y), xytext=(0, 9), textcoords="offset points", ha="center",
                    fontsize=8, color=INK)
    ax.plot(b, y, "o", color=COLORS["bilstm"], ms=9, zorder=2)
    ax.annotate(f"{b:.3f}", (b, y), xytext=(0, 9), textcoords="offset points", ha="center",
                fontsize=8, color=INK)
ax.set_yticks(range(len(CHECKS)), [c[0] for c in CHECKS])
ax.set_ylim(len(CHECKS) - 0.4, -0.7)
ax.set_xlim(0.975, 1.0015)
ax.grid(axis="y", visible=False)
ax.set_xlabel("val macro-F1")
ax.legend(handles=[Line2D([], [], marker="o", ls="", color=COLORS["bilstm"], label="BiLSTM"),
                   Line2D([], [], marker="o", ls="", color=COLORS["tfidf"], label="TF-IDF + LR")],
          loc="center left", bbox_to_anchor=(0.0, 0.37), frameon=False, fontsize=9)
ax.set_title("ISOT checks: removing web clutter changes nothing", loc="left")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "seq_isot_checks.png", dpi=150)

**Takeaway:** removing web clutter (links, photo credits, embed code) doesn't change BiLSTM's score
(0.997 → 0.997), the same as Role A found for TF-IDF. Reading the end of the article (0.999) or adding
the headline (0.998) helps only slightly.
**In plain terms:** the model isn't cheating with leftover web junk. ISOT's real and fake articles
differ in topic and writing style, so they're easy to separate without checking any facts.

## 5. Tables (same numbers as the charts)

In [ ]:
#Phase 1 table
p1 = phase1.assign(score=[r[METRIC[r["dataset"]]] for _, r in phase1.iterrows()])
table1 = p1.pivot(index="model", columns="dataset", values="score")[[t for t, _ in TASKS]]
table1.loc["tfidf (baseline)"] = [tfidf.loc[t, METRIC[t]] for t, _ in TASKS]
table1.round(3)

In [ ]:
#Phase 2 table: every BiLSTM sweep / check run
cols = {"notes": "group", "dataset": "dataset", "params.max_len": "max_len", "params.truncation": "truncation",
        "params.hidden_size": "hidden", "params.dropout": "dropout", "params.freeze_embeddings": "frozen",
        "f1_macro": "val_f1", "params.best_epoch": "best_epoch", "runtime_s": "runtime_s"}
phase2 = seq[(seq["model"] == "bilstm") & seq["notes"].str.match("sens:|check:")]
phase2[list(cols)].rename(columns=cols).sort_values(["group", "dataset"]).round(3).reset_index(drop=True)

## Summary
1. **BiLSTM is the best sequential model** on 3 of 4 tasks, and word order clearly helps on COVID.
2. **LIAR is insensitive to tuning**: the limit is the information in a short claim, not the model.
3. **ISOT: only max length matters**, and 512 words is the sweet spot for score vs time.
4. **ISOT's near-perfect score is not from web clutter**; it comes from topic and style differences.